In [ ]:
!pip install -q scispacy
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

print("Medical NLP Models Installed.")

In [ ]:
import spacy
import scispacy
import pandas as pd
import re
from tqdm.auto import tqdm

try:
    nlp = spacy.load("en_core_sci_sm")
    print("Medical NLP Model Loaded Successfully.")
except:
    print("Model not found. Please run the installation cell first.")

def final_clean_extractor(text):
    if not text or len(str(text)) < 3:
        return []
    
    doc = nlp(str(text))
    extracted = []
    
    noise_words = {
        "medication", "dose", "tablet", "capsule", "name", "history", 
        "patient", "daily", "mouth", "sodium", "chloride", "water", 
        "admission", "list", "accurate", "status", "none", "sig"
    }
    
    for ent in doc.ents:
        name = ent.text

        name = re.sub(r'\d+(\.\d+)?\s*(mg|ml|mcg|g|%|units|puff|tab|cap|mEq|intl|unit)', '', name, flags=re.IGNORECASE)

        name = re.sub(r'\b(PO|IV|IM|SC|QD|BID|TID|QHS|PRN|DAILY|ORAL|MOUTH|BY|ADMISSION)\b', '', name, flags=re.IGNORECASE)

        name = re.sub(r'[^\w\s]', '', name)
        name = re.sub(r'\s+', ' ', name).strip().title()

        if len(name) > 3 and name.lower() not in noise_words and not name.isdigit():
            extracted.append(name)
            
    return list(set(extracted))

print("Starting final extraction on 100 records...")
tqdm.pandas()
df_100['processed_generic_meds'] = df_100['home_meds_raw'].progress_apply(final_clean_extractor)

final_output = "gold_100_final_standardized.parquet"
df_100.to_parquet(final_output, index=False)

print(f" Mission Accomplished! File saved as: {final_output}")

display(df_100[df_100['processed_generic_meds'].map(len) > 0][['home_meds_raw', 'processed_generic_meds']].head(10))

In [ ]:
import pandas as pd
import os
import shutil
from IPython.display import FileLink

INPUT_FILE = "gold_100_final_standardized.parquet" 
OUTPUT_FOLDER = "Patients_Individual_Files"
COLUMNS_TO_KEEP = ['subject_id', 'diagnosis_names', 'processed_generic_meds']

if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)
    print(f"Created folder: {OUTPUT_FOLDER}")

if os.path.exists(INPUT_FILE):
    df = pd.read_parquet(INPUT_FILE)

    existing_cols = [col for col in COLUMNS_TO_KEEP if col in df.columns]
    df_subset = df.head(20)[existing_cols]
    
    print(f"Processing {len(df_subset)} individual patient files...")

    for index, row in df_subset.iterrows():
        patient_df = pd.DataFrame([row])

        subject_id = str(row['subject_id'])
        file_name = f"{OUTPUT_FOLDER}/patient_{subject_id}.parquet"

        patient_df.to_parquet(file_name, index=False)
        print(f"Saved: {file_name}")

    zip_name = "Individual_Patients_Data"
    shutil.make_archive(zip_name, 'zip', OUTPUT_FOLDER)
    
    print("\n" + "="*30)
    print("Task Completed Successfully!")
    print(f"Each file contains columns: {existing_cols}")
    print("="*30)

    display(FileLink(f"{zip_name}.zip"))

else:
    print(f"Error: File '{INPUT_FILE}' not found. Please check the file name.")